In [3]:
import os
import requests
import json
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # finds GenAI/LLM/.env by searching parent directories
OPEN_ROUTER_API_KEY = os.environ["OPEN_ROUTER_API_KEY"]

# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {OPEN_ROUTER_API_KEY}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "openai/gpt-6-luna-pro",
    "messages": [
        {
          "role": "user",
          "content": "How many r's are in the word 'strawberry'?"
        }
      ],
    "reasoning": {"enabled": True}
  })
)

# Extract the assistant message with reasoning_details
response = response.json()
if 'choices' not in response:
    raise RuntimeError(f"OpenRouter request failed: {response}")
response = response['choices'][0]['message']

# Preserve the assistant message with reasoning_details
messages = [
  {"role": "user", "content": "How many r's are in the word 'strawberry'?"},
  {
    "role": "assistant",
    "content": response.get('content'),
    "reasoning_details": response.get('reasoning_details')  # Pass back unmodified
  },
  {"role": "user", "content": "Are you sure? Think carefully."}
]

# Second API call - model continues reasoning from where it left off
response2 = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {OPEN_ROUTER_API_KEY}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "openai/gpt-6-luna-pro",
    "messages": messages,  # Includes preserved reasoning_details
    "reasoning": {"enabled": True}
  })
)
response2 = response2.json()
if 'choices' not in response2:
    raise RuntimeError(f"OpenRouter request failed: {response2}")
print(response2['choices'][0]['message']['content'])

Yes—“strawberry” has 3 r’s: one after “st” and two at the end.
